# Module 7 : L'Exploration par BFS pour configurations arbitraires 🗺️

Jusqu'ici, l'algorithme récursif classique ne fonctionne que si tous les disques partent du piquet A.
Mais que faire si la configuration de départ est **totalement mélangée** ?

Ex : Disque 1 sur C, Disque 2 sur A, Disque 3 sur B → comment trouver le chemin le plus court ?

La réponse : le **BFS** (Breadth-First Search — Recherche en Largeur).

## 1. Représenter un état comme un tuple

Chaque configuration du jeu est un **état** : la position de chaque disque sur les trois piquets.
On représente un état par un tuple de 3 tuples : `(contenu_A, contenu_B, contenu_C)`.

Exemple : disques 3, 2, 1 tous sur A → `((3, 2, 1), (), ())`
(Le premier élément de chaque tuple est le fond du piquet, le dernier est le sommet.)

In [ ]:
# État initial : tous les disques sur A
etat_initial_3 = ((3, 2, 1), (), ())

# État final : tous les disques sur C
etat_final_3 = ((), (), (3, 2, 1))

# Un état intermédiaire après 1 coup (disque 1 de A vers C) :
etat_apres_1_coup = ((3, 2), (), (1,))

print("État initial :", etat_initial_3)
print("État final   :", etat_final_3)
print("Après 1 coup :", etat_apres_1_coup)

## 🧩 Exercice 1 : Créer les états initiaux et finaux (★☆☆)

Créez deux fonctions :
- `etat_initial(n)` : retourne l'état où tous les n disques sont sur le piquet A (tuple de tuple)
- `etat_final(n)` : retourne l'état où tous les n disques sont sur le piquet C

**Rappel :** `tuple(range(n, 0, -1))` crée `(n, n-1, ..., 1)` — les disques du plus grand au plus petit.

In [ ]:
def etat_initial(n):
    # Tous les disques sur le piquet A (index 0), du plus grand au plus petit
    return (tuple(range(n, 0, -1)), (), ())

def etat_final(n):
    # Tous les disques sur le piquet C (index 2)
    return ...   # À compléter

print(etat_initial(3))   # → ((3, 2, 1), (), ())
print(etat_final(3))     # → ((), (), (3, 2, 1))

In [ ]:
assert etat_initial(3) == ((3, 2, 1), (), ())
assert etat_final(3) == ((), (), (3, 2, 1))
assert etat_initial(1) == ((1,), (), ())
assert etat_final(1) == ((), (), (1,))
print("✅ Exercice 1 validé !")

## 2. Générer les états voisins

Depuis un état donné, un "voisin" est un état atteignable en **un seul coup légal**.
Pour générer tous les voisins, on essaie toutes les paires de piquets (source, destination) et on vérifie si le coup est légal.

Un coup est légal si :
- La source n'est pas vide
- La destination est vide **ou** le sommet de la source est plus petit que le sommet de la destination

## 🧩 Exercice 2 : La fonction `get_voisins(etat)` (★★☆)

Créez une fonction `get_voisins(etat)` qui retourne la **liste de tous les états atteignables** depuis `etat` en un seul coup légal.

Étapes :
1. Bouclez sur les 6 paires de piquets possibles : `(0,1), (0,2), (1,0), (1,2), (2,0), (2,1)`
2. Pour chaque paire `(i, j)`, vérifiez si le coup est légal
3. Si oui, créez le nouvel état (immutable → convertir en listes, modifier, reconvertir en tuples)

In [ ]:
def get_voisins(etat):
    voisins = []
    paires = [(0,1), (0,2), (1,0), (1,2), (2,0), (2,1)]
    
    for i, j in paires:
        source = etat[i]
        dest   = etat[j]
        
        # Vérifier si le coup est légal
        if len(source) == 0:
            continue   # Source vide, impossible
        
        disque = source[-1]   # Sommet de la source
        
        if len(dest) > 0 and disque > dest[-1]:
            continue   # Coup illégal : grand disque sur petit
        
        # Créer le nouvel état (les tuples sont immutables, on convertit en listes)
        nouvel_etat = list(list(p) for p in etat)
        nouvel_etat[i].pop()
        nouvel_etat[j].append(disque)
        
        voisins.append(tuple(tuple(p) for p in nouvel_etat))
    
    return voisins

# Test :
etat = ((3, 2, 1), (), ())
voisins = get_voisins(etat)
print(f"Depuis l'état initial, {len(voisins)} voisin(s) :")
for v in voisins:
    print("  ", v)

In [ ]:
etat = ((3, 2, 1), (), ())
voisins = get_voisins(etat)
assert len(voisins) == 2, f"Depuis l'état initial (1 disque disponible), il y a 2 voisins (vers B ou C), pas {len(voisins)}."
assert ((3, 2), (), (1,)) in voisins
assert ((3, 2), (1,), ()) in voisins
print("✅ Exercice 2 validé !")

## 3. L'algorithme BFS

Le BFS explore les états **couche par couche** : d'abord tous les états à 1 coup, puis à 2 coups, etc.
Il utilise une **File** (FIFO) : on enfile les états à explorer et on défile dans l'ordre d'arrivée.
On garde aussi un ensemble `etats_visites` pour ne jamais repasser par le même état.

```
File : [état_initial]
Tant que la file n'est pas vide :
    Dépiler l'état courant
    Si c'est l'état final → on a trouvé le chemin !
    Sinon → enfiler tous ses voisins non encore visités
```

## 🧩 Exercice 3 : Compléter l'algorithme BFS (★★★)

La structure de la fonction `bfs_hanoi` est donnée. Complétez la boucle `while`.

À chaque itération :
1. Dépiler `(etat_courant, chemin_courant)` de la file
2. Si `etat_courant == etat_final` → retourner `chemin_courant`
3. Sinon → pour chaque voisin non visité, enfiler `(voisin, chemin_courant + [voisin])`

La fonction retourne la liste des états du chemin (solution), ou `None` si aucune solution.

In [ ]:
from collections import deque

def bfs_hanoi(etat_depart, etat_destination):
    file = deque([(etat_depart, [etat_depart])])
    visites = {etat_depart}
    
    while file:
        # 1. Dépiler l'état courant
        etat_courant, chemin = file.popleft()
        
        # 2. Vérifier si on a trouvé la solution
        if etat_courant == etat_destination:
            return chemin
        
        # 3. Explorer les voisins
        for voisin in get_voisins(etat_courant):
            if voisin not in visites:
                visites.add(voisin)
                file.append((voisin, chemin + [voisin]))
    
    return None   # Aucune solution trouvée

# Test :
depart = etat_initial(3)
arrivee = etat_final(3)
solution = bfs_hanoi(depart, arrivee)
print(f"Solution trouvée en {len(solution) - 1} coups :")
for i, etat in enumerate(solution):
    print(f"  Étape {i} : {etat}")

In [ ]:
solution = bfs_hanoi(etat_initial(3), etat_final(3))
assert solution is not None, "Une solution doit exister !"
assert len(solution) - 1 == 7, f"La solution optimale pour 3 disques fait 7 coups, pas {len(solution)-1}."
assert solution[0] == etat_initial(3)
assert solution[-1] == etat_final(3)
print("✅ Exercice 3 validé ! Le BFS trouve le chemin optimal.")

## 🧩 Exercice 4 : L'explosion combinatoire (★★★★)

Modifiez `bfs_hanoi` pour qu'elle retourne aussi le **nombre d'états explorés**.
Puis comparez ce nombre avec `3^n` (le nombre total d'états possibles) pour n = 2, 3, 4, 5.

Tracez les résultats avec matplotlib.

In [ ]:
import matplotlib.pyplot as plt

def bfs_hanoi_stats(etat_init, etat_fin):
    """Retourne (chemin, nb_etats_explores)."""
    file = deque([(etat_init, [etat_init])])
    visites = {etat_init}
    
    while file:
        etat_courant, chemin = file.popleft()
        if etat_courant == etat_fin:
            return chemin, len(visites)
        for voisin in get_voisins(etat_courant):
            if voisin not in visites:
                visites.add(voisin)
                file.append((voisin, chemin + [voisin]))
    
    return None, len(visites)

valeurs_n = [2, 3, 4, 5]
etats_explores = []
etats_possibles = []

for n in valeurs_n:
    _, nb = bfs_hanoi_stats(etat_initial(n), etat_final(n))
    etats_explores.append(nb)
    etats_possibles.append(3**n)
    print(f"n={n} : {nb} états explorés / {3**n} états possibles ({100*nb/3**n:.1f}%)")

plt.figure(figsize=(8, 5))
plt.plot(valeurs_n, etats_possibles, 'r--o', label='3^n états possibles')
plt.plot(valeurs_n, etats_explores, 'b-o', label='États explorés par BFS')
plt.xlabel("Nombre de disques (n)")
plt.ylabel("Nombre d'états")
plt.title("BFS vs explosion combinatoire")
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()